# arms · Outcomes  `[EVAL]`

Full-conversation eval outcomes (the held-out measure, never the training reward) — **the global scores** (one number per conversation per questionnaire), for **all four arms on one axis** (PTO K=0/K=5, GRPO K=0/K=5). The all-metric trajectory grid (THE main figure), a per-metric learning-curve catalog (`trajectories/`), and every endpoint artifact as a **final + best pair** (best = each arm's peak iteration on its own training oracle — the checkpoint you would actually select; on the primary oracle GRPO K=0 peaks at iter 8 then regresses). Exports → `results/arms/outcomes/{figures,tables}/<judge>/`; the scorecard cells are re-issued as a citable ledger `tables/<judge>/outcomes_numbers.json`. (The duplicate `headline/` presentation copies were retired 2026-08-26 — curation now lives in `results/README.md`, which links the canonical artifacts.)

**Grader.** This family is rendered once per grader (`EDA_JUDGE`): the primary oracle (gpt-4o-mini, the training reward) and the held-out judge (claude-haiku-4-5). Every artifact below is produced by ONE grader — the one named in its path. **Endpoints.** Each arm's "final" is its own last SCORED iteration under the grader being rendered (derived per render, printed in every caption's support line — all four arms currently reach the same one); "best" = each arm's peak Q1+Q2 iteration (its own training rubric) under the grader being rendered — on the primary leaf that is the own training oracle; on the held-out leaf it is the held-out judge's pick, so the selected iterations can differ between the two leaves. **Pairing unit** for every vs-base contrast is the persona (96 per model state).

Drill deeper: per-questionnaire items/subscales → `arms/questionnaires`, validity + reward-hacking → `arms/validity`, persona splits → `arms/heterogeneity`, stats tables → `arms/stats`; the K contrast lives in `lookahead/`, the method contrast in `method/`.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, stats
# FAMILY = which results folder this notebook owns; JUDGE = which grader's scores are read
# ("" = the primary oracle; render_results.py sets EDA_JUDGE per grader on disk).
cfg = eda_analysis.EdaConfig(family="arms/outcomes", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables for the active judge (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the leaf's _provenance.md (reset just removed the one notebook_setup wrote)

# Best iteration per arm (own training oracle — the checkpoint you'd select); drives the *_best artifacts.
BEST = eda_analysis.best_iteration_by_arm(S.SCORES)
FINAL = {a: int(g.loc[~g.is_base, "iteration"].max()) for a, g in S.SCORES.groupby("arm")}
print("final iteration per arm:", FINAL)
print("best iteration per arm: ", BEST)
from eda_analysis.constants import judge_dirname
GRADER = judge_dirname(S.JUDGE)          # short grader label named in every caption (gpt-4o-mini | claude-haiku-4-5)
# Censoring is DERIVED from the frame in hand - FINAL above already knows where each arm stops -
# and it is grader-dependent: the score lake covers an arm further under one judge than another, so
# a written-down iteration is wrong for at least one leaf the moment a run advances.
CENSOR = eda_analysis.support_note(S.SCORES, subject=f"no later state scored by {GRADER}")
BEST_BY = f"peak Q1+Q2 (own training rubric) under {GRADER}"   # how *_best artifacts pick each arm's iteration


## 1 · Outcome trajectories — all metrics  `[EVAL]`
Per-metric mean ± 95% CI across iterations, all four arms overlaid — the one-glance overview. Global-eval (halo) rubrics beside PCT and MICI ↓ (lower = better); caveat under the grid. Each arm's line ends at its last scored iteration under this grader; all four currently reach the same one, and the caption's support line names any arm that does not.

In [ ]:
fig = plotting.trajectory_grid(S.SCORES, palette=S.PALETTE, arms=cfg.focus_arms)
_cap = (f"Full-conv eval on {GRADER}: per-metric mean +/- 95% CI across iterations, all four arms overlaid "
        "(all 9 evaluation metrics; MICI is lower-is-better). N=96 personas per model state; " + CENSOR)
exports.save_fig(fig, "trajectories_all_metrics", caption="THE main per-arm figure. " + _cap)
plt.show()

## 2 · Per-metric learning curves  `[EVAL]`
One full-size curve per metric → `figures/<judge>/trajectories/`. Peaks that precede the final iteration are auto-flagged ("peak → regresses" — GRPO K=0's iter-8 peak) on higher-is-better metrics only; on MICI ↓ the maximum is the *worst* iteration, so no such flag is drawn there. The oracle-noise band draws only on Q1+Q2 (the ~0.10 reproducibility figure was measured on that rubric).

In [ ]:
for m in S.METRICS:
    fig = plotting.single_metric_trajectory(S.SCORES, m, palette=S.PALETTE, arms=cfg.focus_arms,
                                            oracle_noise=(S.ORACLE_NOISE if m == "Q1Q2" else None),
                                            mark_peaks=True)
    lower = m in eda_analysis.LOWER_IS_BETTER
    exports.save_fig(
        fig, f"trajectory_{m}", group="trajectories",
        caption=f"{eda_analysis.display_label(m)} across iterations per arm on {GRADER} (mean +/- 95% CI, N=96 personas), all four arms; "
                + ("lower = better; no peak flag on a lower-is-better metric. " if lower else
                   "dotted vline flags a peak that precedes the final iteration (regression). ")
                + (f"Grey band = oracle reproducibility (~{S.ORACLE_NOISE:.2f}) around base. " if m == "Q1Q2" else "")
                + CENSOR)
    plt.show()

## 3 · Did it work? — effect vs base, FINAL + BEST  `[EVAL]`
The vs-base effect per arm × metric as a forest plot (all 9 metrics; MI-Inconsistency ↓ rows are valence-inverted — red = moved the wrong way), reported twice: at each arm's **final** iteration (its own last scored iteration under this grader) and at each arm's **best** iteration (peak Q1+Q2 under the grader being rendered — the setup cell prints `BEST` per arm; the honest model-selection view). Persona-paired throughout (Δ = target − own base, 96 pairs; dz + Wilcoxon, Holm across rubrics within arm). Full stats tables in `arms/stats`.

In [ ]:
MR_ALL = {}
for target, iters_note in (("final", "each arm's FINAL iteration"),
                           ("best", "each arm's BEST iteration (" + BEST_BY + ")")):
    MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target=target), S.SCORES)
    MR_ALL[target] = MR
    fig = plotting.effect_forest(MR, title=f"Effect on full-conversation eval vs base — {iters_note}")
    _cap = (f"Improvement vs own base (Δ + 95% bootstrap CI) per arm x rubric at {iters_note}, "
            f"full-conv eval on {GRADER}; persona-paired (96 pairs), dot color = effect-size label, dz annotated; "
            "MICI ↓ rows valence-inverted (red = worse). " + CENSOR)
    exports.save_fig(fig, f"effect_vs_base_forest_{target}", caption=_cap)
    plt.show()

## 4 · Endpoint bars — FINAL + BEST models  `[EVAL]`
Per-metric bars over the selected checkpoints only (the four arm-bases pooled into one descriptive `Base`, dotted base line): once at each arm's **final** iteration, once at its **best** (own-oracle) iteration. The all-iteration story lives in the trajectory grid (§1), so no exhaustive every-model wall here.

In [ ]:
for target, select, note in (("final", eda_analysis.final_per_experiment, "final iteration"),
                             ("best", eda_analysis.best_per_experiment, "best iteration (" + BEST_BY + ")")):
    D = eda_analysis.collapse_base(select(S.SCORES)[0])
    fig = plotting.outcomes_by_model(D, palette=plotting.arm_palette(sorted(D.arm.unique())),
                                     order=plotting.model_order(D),
                                     title=f"Outcome metrics at each arm's {note} — full-conversation eval (dotted = base)")
    exports.save_fig(fig, f"outcomes_by_model_{target}",
                     caption=f"Each of the four arms at its {note} x metrics on {GRADER}; mean +/- 95% CI over 96 personas "
                             "(the arm-bases pooled into Base; dotted line = base; MICI lower-is-better). " + CENSOR)
    plt.show()

## 5 · Scorecard — FINAL + BEST, every evaluation metric  `[EVAL]`
Each arm's score per metric at its **final** and **best** iteration (one table, `target` column): the global-eval (halo) rubrics beside `PCT`, `MICI ↓`, and the derived `R:Q`/`%CR`/`%MICO`. **Read:** the halo cluster can rise while technique / patient-outcome lag — the multi-skill story the global-eval rubrics hide; the final-vs-best gap is GRPO K=0's post-peak regression (primary oracle). A `.json` ledger of the same cells (`outcomes_numbers.json`, keys `<target>.<arm>.<metric>`) sits beside the table for papers/decks to cite.

In [ ]:
LB = pd.concat([plotting.leaderboard_scorecard(S.SCORES, selection="final").assign(target="final"),
                plotting.leaderboard_scorecard(S.SCORES, selection="best").assign(target="best")],
               ignore_index=True)
LB = LB[["target"] + [c for c in LB.columns if c != "target"]]
display(LB)
_cap = (f"Final AND best iteration per arm (target column) on {GRADER}: every evaluation metric side by side "
        "(means over 96 personas; MICI lower-is-better, flagged ↓; best = " + BEST_BY + "). " + CENSOR)
exports.save_table(LB, "leaderboard_scorecard", caption=_cap)

# Number ledger: <target>.<arm>.<metric> -> value (source = the table above), so a paper/deck can cite a cell by key.
# Named outcomes_numbers (not leaderboard_scorecard) so its CAPTIONS line doesn't replace the table's.
_num = {}
for _, r in LB.iterrows():
    arm_key = str(r["arm"]).replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    for c in LB.columns:
        if c in ("target", "arm"):
            continue
        val = r[c]
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        _num[f"{r['target']}.{arm_key}.{c}"] = {
            "value": (int(val) if c == "iteration" else float(val)),
            "source": f"tables/{GRADER}/leaderboard_scorecard.md",
            "note": f"{r['target']} iteration of {r['arm']} on {GRADER}; " + ("lower = better" if "↓" in c else "higher = better"),
        }
exports.save_numbers("outcomes_numbers", _num,
                     caption=f"Ledger of the scorecard cells (keys <target>.<arm>.<metric>; best = {BEST_BY}) on {GRADER}; " + CENSOR)

## 6 · Artifact index
Drop stale captions, then refresh `results/arms/INDEX.md` + the root `results/INDEX.md` (every notebook ends with this — whichever ran last completes the map).

In [ ]:
print("orphan captions removed:", exports.prune_orphan_captions())
print("index ->", exports.build_index())